# 🚀 Phase 5 — Démonstration Interactive avec Gradio
**Projet PFE — Détection d'Anomalies dans des Objets Industriels 3D**  
**Stagiaire :** Olivier OUEDRAOGO — SUPMTI ISI  
**Entreprise :** 3D SMART FACTORY, Mohammedia  

---
**⚠️ Prérequis : même session que la Phase 3**  
Variables requises : `extractor`, `knn_btf`, `pca`, `THRESHOLD`, `DEVICE`

**Ce que fait cette interface :**
- Upload d'une image RGB + fichier depth (ou sélection depuis le dataset)
- Détection automatique : normale ou défectueuse
- Affichage du score d'anomalie et de la carte de chaleur
- Interface accessible via un lien public Gradio

## 1. Installation de Gradio

In [ ]:
!pip install gradio -q
import gradio as gr
print(f'✅ Gradio {gr.__version__} installé')

## 2. Vérification session Phase 3

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import tifffile
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torchvision.transforms as T
from PIL import Image
from pathlib import Path
from sklearn.preprocessing import normalize as sk_normalize
import io, os

checks = {
    'extractor' : 'extractor' in dir(),
    'knn_btf'   : 'knn_btf'   in dir(),
    'pca'       : 'pca'       in dir(),
    'THRESHOLD' : 'THRESHOLD' in dir(),
    'DEVICE'    : 'DEVICE'    in dir(),
}
all_ok = all(checks.values())
for name, ok in checks.items():
    print(f'  {"✅" if ok else "❌"}  {name}')

if all_ok:
    print(f'\n✅ Session active — THRESHOLD={THRESHOLD:.6f} | DEVICE={DEVICE}')
else:
    print('\n❌ Session expirée — relancez le notebook Phase 3 complet')

rgb_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225])
])
print('✅ Transformation RGB prête')

## 3. Fonctions de prédiction

In [ ]:
def preprocess_rgb(pil_image):
    """Transforme une image PIL RGB en tenseur normalisé."""
    return rgb_transform(pil_image.convert('RGB'))


def preprocess_depth_from_tiff(tiff_path):
    """
    Charge un fichier .tiff MVTec et extrait la carte de profondeur Z.
    Retourne un tenseur normalisé et une image PIL pour affichage.
    """
    xyz_map = tifffile.imread(str(tiff_path))
    z_map   = xyz_map[:, :, 2].astype(np.float32)
    z_map   = (z_map - z_map.min()) / (z_map.max() - z_map.min() + 1e-8)
    z_pil   = Image.fromarray((z_map * 255).astype('uint8')).convert('RGB')
    z_tensor = rgb_transform(z_pil)
    return z_tensor, z_pil


def preprocess_depth_from_image(pil_image):
    """
    Utilise une image depth uploadée directement (niveaux de gris).
    """
    gray    = pil_image.convert('L')
    arr     = np.array(gray).astype(np.float32)
    arr     = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    z_pil   = Image.fromarray((arr * 255).astype('uint8')).convert('RGB')
    z_tensor = rgb_transform(z_pil)
    return z_tensor, z_pil


def compute_anomaly_map_from_tensors(rgb_tensor, depth_tensor):
    """
    Calcule la carte d'anomalie spatiale (28×28) depuis deux tenseurs.
    """
    extractor.eval()
    with torch.no_grad():
        rgb   = rgb_tensor.unsqueeze(0).to(DEVICE)
        depth = depth_tensor.unsqueeze(0).to(DEVICE)

        r1, r2, r3 = extractor(rgb)
        d1, d2, d3 = extractor(depth)

        r1_r = F.interpolate(r1, size=(28,28), mode='bilinear', align_corners=False)
        r3_r = F.interpolate(r3, size=(28,28), mode='bilinear', align_corners=False)
        d1_r = F.interpolate(d1, size=(28,28), mode='bilinear', align_corners=False)
        d3_r = F.interpolate(d3, size=(28,28), mode='bilinear', align_corners=False)

        combined   = torch.cat([r1_r, r2, r3_r, d1_r, d2, d3_r], dim=1)
        _, C, H, W = combined.shape
        patches    = combined[0].permute(1,2,0).reshape(-1, C).cpu().numpy()
        patches    = sk_normalize(patches)
        patches_pca = pca.transform(patches)

        dists, _   = knn_btf.kneighbors(patches_pca)
        scores     = dists.mean(axis=1)
        heatmap    = scores.reshape(H, W)

    score = float(heatmap.max())
    return heatmap, score


def make_heatmap_overlay(pil_rgb, heatmap, alpha=0.5):
    """
    Superpose la heatmap sur l'image RGB.
    Retourne une image PIL RGBA.
    """
    rgb_224 = np.array(pil_rgb.resize((224, 224)))

    # Upscale heatmap → 224×224
    hmap_up = np.array(Image.fromarray(heatmap).resize((224, 224), Image.BILINEAR))
    hmin, hmax = hmap_up.min(), hmap_up.max()
    hmap_norm  = (hmap_up - hmin) / (hmax - hmin + 1e-8)

    # Colormap jet → RGBA
    colormap   = cm.get_cmap('jet')
    hmap_color = (colormap(hmap_norm)[:, :, :3] * 255).astype(np.uint8)

    # Blend
    overlay = (rgb_224 * (1 - alpha) + hmap_color * alpha).astype(np.uint8)
    return Image.fromarray(overlay)


def make_heatmap_image(heatmap):
    """
    Génère l'image de la heatmap seule (RdYlGn_r).
    """
    hmin, hmax = heatmap.min(), heatmap.max()
    hmap_norm  = (heatmap - hmin) / (hmax - hmin + 1e-8)
    hmap_up    = np.array(Image.fromarray(hmap_norm).resize((224, 224), Image.BILINEAR))
    colormap   = cm.get_cmap('RdYlGn_r')
    hmap_color = (colormap(hmap_up)[:, :, :3] * 255).astype(np.uint8)
    return Image.fromarray(hmap_color)


print('✅ Fonctions de prédiction définies')

## 4. Préparation des exemples du dataset (sélection rapide)

In [ ]:
# Préparer des exemples pré-chargés depuis le dataset
# pour la démonstration rapide en soutenance

EXTRACT_DIR = '/content/mvtec3d'
CATEGORY    = 'bagel'
CAT_PATH    = Path(EXTRACT_DIR) / CATEGORY

def collect_demo_samples(cat_path, n_per_type=3):
    """Collecte quelques exemples par type pour la démo."""
    samples = {}
    test_path = cat_path / 'test'
    for defect_dir in sorted(test_path.iterdir()):
        if not defect_dir.is_dir():
            continue
        dtype   = defect_dir.name
        xyz_dir = defect_dir / 'xyz'
        rgb_dir = defect_dir / 'rgb'
        if not xyz_dir.exists():
            continue
        files = sorted(xyz_dir.glob('*.tiff'))[:n_per_type]
        samples[dtype] = [
            {
                'xyz' : str(f),
                'rgb' : str(rgb_dir / f.name.replace('.tiff', '.png')),
                'label': dtype
            }
            for f in files
        ]
    return samples

demo_samples = collect_demo_samples(CAT_PATH, n_per_type=3)

# Construire la liste des choix pour le dropdown
example_choices = []
for dtype, items in demo_samples.items():
    for i, item in enumerate(items):
        label = f"{dtype} #{i+1}"
        example_choices.append(label)

# Mapping label → chemins
example_map = {}
for dtype, items in demo_samples.items():
    for i, item in enumerate(items):
        label = f"{dtype} #{i+1}"
        example_map[label] = item

print(f'✅ {len(example_choices)} exemples prêts pour la démo')
for choice in example_choices:
    print(f'   {choice}')

## 5. Interface Gradio — Démonstration interactive

In [ ]:
def predict_from_dataset(example_label):
    """
    Prédit l'anomalie depuis un exemple du dataset sélectionné.
    Retourne : image RGB, heatmap, superposition, résultat textuel
    """
    if example_label not in example_map:
        return None, None, None, 'Exemple non trouvé'

    item = example_map[example_label]

    # Charger RGB
    try:
        pil_rgb = Image.open(item['rgb']).convert('RGB')
    except:
        return None, None, None, 'Erreur chargement RGB'

    # Charger depth depuis .tiff
    try:
        depth_tensor, depth_pil = preprocess_depth_from_tiff(item['xyz'])
    except:
        return None, None, None, 'Erreur chargement depth'

    rgb_tensor = preprocess_rgb(pil_rgb)

    # Prédiction
    heatmap, score = compute_anomaly_map_from_tensors(rgb_tensor, depth_tensor)
    is_defect      = score > THRESHOLD

    # Images de sortie
    heatmap_img = make_heatmap_image(heatmap)
    overlay_img = make_heatmap_overlay(pil_rgb, heatmap, alpha=0.45)

    # Résultat textuel
    verdict = '⚠️ DÉFAUT DÉTECTÉ' if is_defect else '✅ PIÈCE NORMALE'
    correct = ((is_defect and item['label'] != 'good') or
               (not is_defect and item['label'] == 'good'))

    result_text = (
        f"Type réel      : {item['label']}\n"
        f"Score anomalie : {score:.6f}\n"
        f"Seuil décision : {THRESHOLD:.6f}\n"
        f"─────────────────────────\n"
        f"DÉCISION       : {verdict}\n"
        f"RÉSULTAT       : {'[CORRECT]' if correct else '[ERREUR]'}"
    )

    return pil_rgb.resize((224, 224)), heatmap_img, overlay_img, result_text


def predict_from_upload(rgb_image, depth_image):
    """
    Prédit depuis des images uploadées par l'utilisateur.
    rgb_image   : PIL image RGB
    depth_image : PIL image depth (niveaux de gris ou RGB)
    """
    if rgb_image is None:
        return None, None, None, 'Veuillez uploader une image RGB'

    rgb_tensor = preprocess_rgb(rgb_image)

    if depth_image is not None:
        depth_tensor, _ = preprocess_depth_from_image(depth_image)
    else:
        # Si pas de depth, utiliser une image noire (depth=0)
        depth_tensor = torch.zeros_like(rgb_tensor)

    heatmap, score = compute_anomaly_map_from_tensors(rgb_tensor, depth_tensor)
    is_defect      = score > THRESHOLD

    heatmap_img = make_heatmap_image(heatmap)
    overlay_img = make_heatmap_overlay(rgb_image, heatmap, alpha=0.45)

    verdict = '⚠️ DEFAUT DETECTE' if is_defect else '✅ PIECE NORMALE'
    result_text = (
        f"Score anomalie : {score:.6f}\n"
        f"Seuil décision : {THRESHOLD:.6f}\n"
        f"─────────────────────────\n"
        f"DÉCISION : {verdict}"
    )

    return rgb_image.resize((224, 224)), heatmap_img, overlay_img, result_text


print('✅ Fonctions Gradio définies')

## 6. Lancement de l'interface Gradio

In [ ]:
import gradio as gr

# ── Thème et CSS personnalisés ─────────────────────────────────────
custom_css = """
.result-box textarea {
    font-family: monospace;
    font-size: 14px;
    background: #1e1e1e;
    color: #d4d4d4;
}
.gradio-container { max-width: 1100px !important; }
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css,
    title='Détection d\'Anomalies 3D — PFE SUPMTI'
) as demo:

    # ── En-tête ────────────────────────────────────────────────────
    gr.HTML("""
    <div style="text-align:center; padding:20px 0 10px;">
      <h1 style="font-size:28px; margin:0;">🔍 Détection d'Anomalies dans des Objets Industriels 3D</h1>
      <p style="color:#666; margin:8px 0 0;">
        Projet PFE · SUPMTI ISI · Olivier OUEDRAOGO · 3D SMART FACTORY Mohammedia
      </p>
      <p style="color:#888; font-size:13px; margin:4px 0 0;">
        Approche BTF (Back to Feature) · ResNet18 ImageNet · RGB + Depth
      </p>
    </div>
    """)

    # ── Métriques ──────────────────────────────────────────────────
    gr.HTML(f"""
    <div style="display:flex; gap:12px; justify-content:center;
                margin:0 0 20px; flex-wrap:wrap;">
      <div style="background:#d4f1d4; border-radius:8px; padding:10px 20px; text-align:center;">
        <div style="font-size:22px; font-weight:600; color:#1a6b1a;">0.9514</div>
        <div style="font-size:12px; color:#2d8c2d;">AUC-ROC</div>
      </div>
      <div style="background:#d4f1d4; border-radius:8px; padding:10px 20px; text-align:center;">
        <div style="font-size:22px; font-weight:600; color:#1a6b1a;">0.9392</div>
        <div style="font-size:12px; color:#2d8c2d;">F1-Score</div>
      </div>
      <div style="background:#d4f1d4; border-radius:8px; padding:10px 20px; text-align:center;">
        <div style="font-size:22px; font-weight:600; color:#1a6b1a;">0.9140</div>
        <div style="font-size:12px; color:#2d8c2d;">Précision</div>
      </div>
      <div style="background:#d4f1d4; border-radius:8px; padding:10px 20px; text-align:center;">
        <div style="font-size:22px; font-weight:600; color:#1a6b1a;">0.9659</div>
        <div style="font-size:12px; color:#2d8c2d;">Rappel</div>
      </div>
      <div style="background:#fff3cd; border-radius:8px; padding:10px 20px; text-align:center;">
        <div style="font-size:22px; font-weight:600; color:#856404;">bagel</div>
        <div style="font-size:12px; color:#856404;">Catégorie</div>
      </div>
    </div>
    """)

    # ── Deux onglets ───────────────────────────────────────────────
    with gr.Tabs():

        # ── Onglet 1 : Exemples du dataset ─────────────────────────
        with gr.TabItem('📦 Exemples du dataset MVTec 3D-AD'):
            gr.Markdown("""
            Sélectionnez une pièce du dataset de test pour voir la détection en temps réel.
            Les exemples couvrent tous les types de défauts : `good`, `crack`, `hole`,
            `contamination`, `combined`.
            """)

            with gr.Row():
                dropdown = gr.Dropdown(
                    choices=example_choices,
                    label='Choisir une pièce',
                    value=example_choices[0] if example_choices else None
                )
                btn_dataset = gr.Button(
                    'Analyser cette pièce',
                    variant='primary',
                    scale=1
                )

            with gr.Row():
                out_rgb1     = gr.Image(label='Image RGB originale',
                                        type='pil', height=250)
                out_heatmap1 = gr.Image(label='Carte d\'anomalie',
                                        type='pil', height=250)
                out_overlay1 = gr.Image(label='Superposition RGB + Heatmap',
                                        type='pil', height=250)

            out_result1 = gr.Textbox(
                label='Résultat de la détection',
                lines=7,
                elem_classes=['result-box']
            )

            btn_dataset.click(
                fn=predict_from_dataset,
                inputs=[dropdown],
                outputs=[out_rgb1, out_heatmap1, out_overlay1, out_result1]
            )

            # Lancer automatiquement au chargement
            demo.load(
                fn=predict_from_dataset,
                inputs=[dropdown],
                outputs=[out_rgb1, out_heatmap1, out_overlay1, out_result1]
            )

        # ── Onglet 2 : Upload custom ────────────────────────────────
        with gr.TabItem('📤 Analyser votre propre image'):
            gr.Markdown("""
            Uploadez votre propre image RGB d'un bagel industriel.  
            La carte de profondeur est optionnelle — sans elle, la détection
            s'appuie uniquement sur la modalité RGB.
            """)

            with gr.Row():
                in_rgb   = gr.Image(
                    label='Image RGB (obligatoire)',
                    type='pil', height=220
                )
                in_depth = gr.Image(
                    label='Carte de profondeur (optionnel)',
                    type='pil', height=220
                )

            btn_upload = gr.Button(
                'Analyser',
                variant='primary'
            )

            with gr.Row():
                out_rgb2     = gr.Image(label='Image analysée',
                                        type='pil', height=250)
                out_heatmap2 = gr.Image(label='Carte d\'anomalie',
                                        type='pil', height=250)
                out_overlay2 = gr.Image(label='Superposition RGB + Heatmap',
                                        type='pil', height=250)

            out_result2 = gr.Textbox(
                label='Résultat',
                lines=5,
                elem_classes=['result-box']
            )

            btn_upload.click(
                fn=predict_from_upload,
                inputs=[in_rgb, in_depth],
                outputs=[out_rgb2, out_heatmap2, out_overlay2, out_result2]
            )

    # ── Pied de page ───────────────────────────────────────────────
    gr.HTML("""
    <div style="text-align:center; margin-top:20px; padding:10px;
                border-top:1px solid #eee; color:#999; font-size:12px;">
      Dataset : MVTec 3D-AD (Bergmann et al., VISAPP 2022) ·
      Approche : BTF (Horwitz & Hoshen, CVPR 2023) ·
      SUPMTI ISI PFE 2025
    </div>
    """)


# ── Lancement ──────────────────────────────────────────────────────
print('🚀 Lancement de l\'interface Gradio...')
print('   Un lien public sera généré ci-dessous')
print('   Partagez ce lien pour la démonstration en soutenance\n')

demo.launch(
    share=True,          # génère un lien public gradio.live
    debug=False,
    show_error=True
)

## 7. Sauvegarde du modèle complet pour réutilisation

Cette cellule sauvegarde tous les composants du système sur Drive  
pour pouvoir relancer la démo sans réentraîner.

In [ ]:
import pickle, os, torch

SAVE_PATH = '/content/drive/MyDrive/Detection_Anomalie_3D/models'
os.makedirs(SAVE_PATH, exist_ok=True)

# Sauvegarder KNN + PCA + THRESHOLD
pipeline = {
    'knn_btf'   : knn_btf,
    'pca'       : pca,
    'threshold' : float(THRESHOLD),
    'category'  : CATEGORY,
    'approach'  : 'BTF RGB+Depth ResNet18'
}

with open(f'{SAVE_PATH}/btf_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

# Sauvegarder les poids de l'extracteur
torch.save(extractor.state_dict(),
           f'{SAVE_PATH}/btf_extractor.pth')

print('✅ Pipeline complet sauvegardé sur Drive :')
print(f'   {SAVE_PATH}/btf_pipeline.pkl  (KNN + PCA + seuil)')
print(f'   {SAVE_PATH}/btf_extractor.pth  (poids ResNet18)')
print(f'\n   Pour recharger :')
print(f'   import pickle, torch')
print(f'   pipeline = pickle.load(open("btf_pipeline.pkl", "rb"))')
print(f'   knn_btf  = pipeline["knn_btf"]')
print(f'   pca      = pipeline["pca"]')
print(f'   THRESHOLD = pipeline["threshold"]')